In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [4]:
df=pd.read_csv("cleaned_customer_data.csv")

In [5]:
df.head(3)

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Middle-aged,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young Adult,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Middle-aged,7


In [6]:
X = df.drop(columns=['subscription_status', 'customer_id'], errors='ignore')
y = df['subscription_status'].map({'Yes': 1, 'No': 0})

In [7]:
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

In [8]:
print(f"Numerical Features ({len(num_cols)}): {num_cols}")
print(f"Categorical Features ({len(cat_cols)}): {cat_cols}")

Numerical Features (5): ['age', 'purchase_amount', 'review_rating', 'previous_purchases', 'purchase_frequency_days']
Categorical Features (12): ['gender', 'item_purchased', 'category', 'location', 'size', 'color', 'season', 'shipping_type', 'discount_applied', 'payment_method', 'frequency_of_purchases', 'age_group']


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [10]:
print(f"\nTraining set shape: {X_train.shape}")
print(f"Testing set shape:  {X_test.shape}")


Training set shape: (3120, 17)
Testing set shape:  (780, 17)


In [11]:
num_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

In [12]:
cat_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

In [14]:
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

print(f"\nPreprocessed X_train matrix shape: {X_train_preprocessed.shape}")


Preprocessed X_train matrix shape: (3120, 144)


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

In [16]:
# Model A: Baseline Logistic Regression
model_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, class_weight='balanced'))
])



In [17]:
# Model B: Random Forest Classifier
model_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'))
])

In [18]:
model_lr.fit(X_train, y_train)
model_rf.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [19]:
y_pred_lr = model_lr.predict(X_test)
y_prob_lr = model_lr.predict_proba(X_test)[:, 1]

In [20]:
y_pred_rf = model_rf.predict(X_test)
y_prob_rf = model_rf.predict_proba(X_test)[:, 1]

In [21]:
def get_metrics(y_true, y_pred, y_prob):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_prob)
    }

In [22]:
results = pd.DataFrame({
    'Logistic Regression': get_metrics(y_test, y_pred_lr, y_prob_lr),
    'Random Forest': get_metrics(y_test, y_pred_rf, y_prob_rf)
}).T

print("=== Model Comparison Performance ===")
print(results.round(4))

=== Model Comparison Performance ===
                     Accuracy  Precision  Recall  F1-Score  ROC-AUC
Logistic Regression    0.8654     0.6677   1.000    0.8008   0.9094
Random Forest          0.8526     0.6714   0.891    0.7658   0.9118


In [23]:
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [24]:
ohe_feature_names = model_rf.named_steps['preprocessor'] \
                             .named_transformers_['cat'] \
                             .named_steps['onehot'] \
                             .get_feature_names_out(cat_cols)

In [25]:
all_feature_names = num_cols + list(ohe_feature_names)

In [26]:
rf_classifier = model_rf.named_steps['classifier']
importances = rf_classifier.feature_importances_

In [27]:
feature_importance_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

In [28]:
print("=== Top 10 Most Important Features ===")
print(feature_importance_df.head(10))

=== Top 10 Most Important Features ===
                     Feature  Importance
126     discount_applied_Yes    0.249307
125      discount_applied_No    0.214460
6                gender_Male    0.065813
5              gender_Female    0.063783
1            purchase_amount    0.025551
3         previous_purchases    0.025109
0                        age    0.024903
2              review_rating    0.020776
4    purchase_frequency_days    0.009116
87                    size_M    0.005295


In [29]:
joblib.dump(model_rf, 'subscription_model_pipeline.pkl')
print("\nSaved trained model pipeline as 'subscription_model_pipeline.pkl'")


Saved trained model pipeline as 'subscription_model_pipeline.pkl'
